In [1]:
from dotenv import load_dotenv
load_dotenv()

True

# State 정의
-> example structure는 추후에 노드에서 structured output으로 넣도록 하면 됨.

In [2]:
from typing import TypedDict, List, Dict, Optional, Annotated
import operator

class AgentState(TypedDict):
    # 1. 입력 컨텍스트
    user_id: int
    location_name: str
    raw_category: str       # 카카오 API 원본 문자열
    category_id: int        # SQL 조회를 위해 매핑된 DB 카테고리 ID
    
    # 2. DB Node가 SQL로 빌드한 상세 컨텍스트 (Candidate Cards)
    candidate_cards: List[dict] 
    """
    Example Structure:
    [
      {
        "user_card_id": 101,
        "card_name": "KB 토심이",
        "last_month_spent": 450000,
        "remaining_limit": 500,
        "tier_thresholds": [
            {"name": "1구간", "amount": 300000},
            {"name": "2구간", "amount": 600000}
        ],
        "benefit_hint": "스타벅스 10% 청구할인"
      }
    ]
    """
    
    # 3. 실시간 현장 결제 이벤트 (Offline Events)
    offline_events: Annotated[List[dict], operator.add]
    """
    Example Structure:
    [
        {
            "brand": "스타벅스", 
            "pay_system": "Npay", 
            "benefit_detail": "50% 적립", 
            "max_benefit": 3000, 
            "d_day": "D-2"
        }
    ]
    """
    
    # 4. RAG Workers가 PDF를 보고 판별한 결과
    rag_analysis: Annotated[List[dict], operator.add]
    """
    Example Structure:
    [
        {
            "card_id": 101,
            "card_name": "KB 토심이",
            "detected_tier": "1구간",
            "benefit_type": "DISCOUNT",
            "benefit_detail": "스타벅스 2,000원 할인",
            "calculated_benefit": 500,
            "hurdle": 10000,
            "is_excluded": False,  # 파이썬에서는 False로 작성해야 합니다.
            "caveats": "사이렌 오더 결제 시 제외, 백화점 입점 매장 제외",
            "evidence": "PDF 4페이지 '스타벅스 혜택' 섹션..."
        }
    ]
    """
    
    # 5. 최종 출력
    final_advice: str

## Node / Tool / Router

In [ ]:
import requests
from bs4 import BeautifulSoup

# 1. 카드고릴라에 API 요청
card_id = "2685"
url = f"https://api.card-gorilla.com:8080/v1/cards/{card_id}"

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Origin": "https://www.card-gorilla.com",
    "Referer": "https://www.card-gorilla.com/",
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    data = response.json()
    
    # 카드 이름 출력
    card_name = data.get("name")
    print(f"✅ 카드명: {card_name}\n")
    
    # 주요 혜택 정제해서 출력
    for benefit in data.get("key_benefit", []):
        title = benefit.get("title")
        raw_info = benefit.get("info", "")
        
        # HTML 태그 제거 및 줄바꿈 정리
        soup = BeautifulSoup(raw_info, "html.parser")
        clean_info = soup.get_text(separator="\n").strip()
        
        print(f"[{title}]")
        print(clean_info)
        print("-" * 30)
else:
    print(f"❌ 실패! 상태 코드: {response.status_code}")
    print("응답 내용:", response.text[:200])

✅ 카드명: WE:SH Travel

[수수료우대]
해외 이용 수수료 총 1.25% 면제
- 국제 브랜드 수수료(1%) 면제
- 해외 서비스 수수료(0.25%) 면제
- 전월 이용실적 조건 및 수수료 면제 한도 없음
- 해외 가맹점 이용 건에 한하며, 
해외 단기카드대출(현금서비스)는 수수료 면제 제외
- 국내외겸용 카드에 한해 서비스 제공
------------------------------
[해외이용]
해외 이용 환율우대 100% (USD기준)
- 해외 이용 전표 매입 시 매매기준율
*
 적용
* 매매기준율 : 외화 매매시 기준이 되는 환전수수료 없는 환율
- 전월 이용실적 조건 및 환율우대 한도 없음
- 환율우대율 100%는 달러(USD) 기준
- 달러(USD) 외 통화로 결제 시, 국제브랜드사가 정한 환율에 의해 달러(USD)로 환산 후 환산된 달러(USD) 금액에 매매기준율을 적용하여 원화(KRW) 금액으로 청구
- 국내외겸용 카드에 한해 서비스 제공
------------------------------
[공항라운지]
공항 라운지 무료
- 전 세계 공항 라운지 무료 이용 (더라운지멤버스)
- 전월 국내 이용실적 30만원 이상 시 제공
- 제공 횟수 : 연 2회 (연 기준: 1월~12월)
- 최초 발급받은 KB WE:SH Travel 카드 사용등록일(KB Pay 등 간편결제 등록 포함)로부터 다음달 말일까지는 전월 국내 이용실적 30만원 미만시에도 제공
- 본인에 한하여 제공하며, 아이디 공유 등 타인양도 불가
- 이용 방법 : '더라운지' 앱 다운로드 및 회원가입 ▶ WE:SH Travel 카드 등록(본인인증) ▶ 라운지 이용 권 발급 ▶ 라운지 안내데스크에 당일 출발 '항공 탑승권', '여권', '라운지 이용권' 제시
- 문의처 : 더라운지 고객센터(국내 ☎ 1811-7437, 해외 ☎ +82-2-2664-7436)
- 이용 가능 라운지 및 사용 방법 등 세부 내용은 '더라운지' 홈페이지(www.theloungemembers.com)에서 확인 가